In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 11.5 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.validation import check_is_fitted
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from itertools import combinations
from sklearn.metrics import roc_auc_score
from typing import List, Union, Optional
import copy
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/heartdisease/Heart_Disease_Prediction.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [4]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
org['source'] = 'original'


In [5]:
NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

In [6]:
# BINS = []
# for q in [5]:
#     for c in HIGH_CARDINALITY:
#         n = f'{c}_{q}_bin'
#         train_bins, bins = pd.qcut(train[c], q=q, labels=False, retbins=True, duplicates='drop')
#         train[n] = train_bins
#         test[n] = pd.cut(test[c], bins=bins, labels=False, include_lowest=True)
#         BINS.append(n)

# print(BINS)
# print('='*30)
# print(len(BINS))

In [7]:
combine = pd.concat([train.drop(columns='id'), test.drop(columns='id'), org], ignore_index=True).reset_index()

In [8]:
for df, name in zip([train, test, org], ['train', 'test', 'original']):
    print(f'NULL VALUE COUNTS FOR {name}:')
    print(df.isnull().sum())
    print('='*30)
    print(f'{name} shape:')
    print(df.shape)
    print('='*30)
    if name == 'train':
        print('General EDA -- TRAIN ONLY', end='\n')
        print(f'Dtypes :', end='\n')
        print(df.dtypes)
        print('='*30)
        print(f'NUMBER OF UNIQUE VALUES :', end='\n')
        print(df.nunique())
        print('='*30)
    

NULL VALUE COUNTS FOR train:
id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
source                     0
dtype: int64
train shape:
(630000, 16)
General EDA -- TRAIN ONLY
Dtypes :
id                           int64
Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR                       int64
Exercise angina              int64
ST depression              float64
Slope of ST                  int64
Number of ves

In [9]:
print(NUMS)
print(HIGH_CARDINALITY)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']


In [10]:
CATS = []
for c in NUMS:
    n = f'{c}_cat'
    combine[n] = combine[c].astype(str).astype('category')
    CATS.append(n)

print(CATS)
print('='*30)
print(len(CATS))

['Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
13


In [11]:
# INTER = []
# for c1, c2 in combinations(CATS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = (combine[c1].astype(str) + '_' + combine[c2].astype(str)).astype('category')
#     INTER.append(n)

# print(INTER)
# print('='*30)
# print(len(INTER))

In [12]:
# INTER1 = []
# for c1, c2 in combinations(NUMS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = combine[c1] * combine[c2]
#     INTER1.append(n)

# print(INTER1)
# print('='*30)
# print(len(INTER1))

In [13]:
# ENC = []

# for c in INTER:
#     n = f'{c}_enc'
#     combine[n], _ = pd.factorize(combine[c])
#     ENC.append(n)

# print(ENC)
# print('='*30)
# print(len(ENC))

In [14]:
train = combine.loc[combine['source']=='train']
test = combine.loc[combine['source']=='test']
org = combine.loc[combine['source']=='original']

In [15]:
# org[CONFIG.TARGET] = org[CONFIG.TARGET].map(class_mapping)
# TE = []
# TE1 = []
# COND_TE = []

# # ALL_CATS = CATS + CATS1 + BINS
# GLOBAL_MEAN = org[config.TARGET].mean()
# # GLOBAL_STD = train_org_n[config.TARGET].std()
# GLOBAL_COUNT = org[config.TARGET].count()

# ALPHA = 10 

# for c in CATS:
#     # if i%5==0: print(i, end='===')
#     for target in [config.TARGET]:
#         # if target=='study_hours':
#         #     if c in CATS:
#         #         tmp_mean = train_org_n.groupby(c)[target].mean()
#         #         tmp_median = train_org_n.groupby(c)[target].median()
#         #         tmp_std = train_org_n.groupby(c)[target].std()
#         #         tmp_min = train_org_n.groupby(c)[target].min()
#         #         tmp_max = train_org_n.groupby(c)[target].max()

#         # else:
#         tmp_mean = org.groupby(c)[target].mean()
#         tmp_count = org.groupby(c)[target].count()
#             # tmp_median = train_org_n.groupby(c)[target].median()
#         tmp_std = org.groupby(c)[target].std()
#             # tmp_min = train_org_n.groupby(c)[target].min()
#             # tmp_max = train_org_n.groupby(c)[target].max()

        
#         # tmp_count = train_org_n.groupby(c)[target].size()
#         # tmp_mean_delta = tmp_mean - GLOBAL_MEAN
#         # tmp_std_delta = tmp_std - GLOBAL_STD
#         # tmp_mean_ratio = tmp_mean / GLOBAL_MEAN
#         # tmp_std_ratio = tmp_std / GLOBAL_STD
#         # tmp_count = train_org_n.groupby(c)[config.TARGET].size()

#         n_mean = f'TE_{c}_{target}_mean'
#         n_count = f'TE_{c}_{target}_count'
#         # n_median = f'TE_{c}_{target}_median'
#         n_std = f'TE_{c}_{target}_std'
#         # n_min = f'TE_{c}_{target}_min'
#         # n_max = f'TE_{c}_{target}_max'
#         # n_count = f'TE_{c}_{target}_count'
#         # n_mean_delta = f'TE_{c}_{target}_mean_delta'
#         # n_std_delta = f'TE_{c}_{target}_std_delta'
#         # n_mean_ratio = f'TE_{c}_{target}_mean_ratio'
#         # n_std_ratio = f'TE_{c}_{target}_std_ratio'

#         # n_c = f'TE_{c}_count'
#         print(f'{n_mean}, {n_count}, {n_std}', end=' ')
#         tmp_mean.name = n_mean
#         # tmp_median.name = n_median
#         tmp_std.name = n_std
#         # tmp_min.name = n_min
#         # tmp_max.name = n_max
#         tmp_count.name = n_count
#         # tmp_mean_delta.name = n_mean_delta
#         # tmp_std_delta.name = n_std_delta
#         # tmp_mean_ratio.name = n_mean_ratio
#         # tmp_std_ratio.name = n_std_ratio
        
    
#         stats = (pd.concat([tmp_mean, tmp_count, tmp_std], axis=1).reset_index().rename(columns={'index': c}))
#         org = org.merge(stats, on=c, how='left')
#         train = train.merge(stats, on=c, how='left')
#         test = test.merge(stats, on=c, how='left')

#         if target==config.TARGET:
#             TE.append(n_mean)
#             # TE.append(n_median)
#             TE.append(n_std)
#             # TE.append(n_min)
#             # TE.append(n_max)
#             TE.append(n_count)
#             # TE.append(n_mean_delta)
#             # TE.append(n_std_delta)
#             # TE.append(n_mean_ratio)
#             # TE.append(n_std_ratio)

#         else:
#             TE1.append(n_mean)
#             # TE.append(n_median)
#             # TE1.append(n_std)
#             # TE1.append(n_min)
#             # TE1.append(n_max)   
#     # TE.append(n_c)


In [16]:
for df in [org, train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

In [17]:
FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
26


In [18]:
# class TargetEncoder(BaseEstimator, TransformerMixin):
#     def __init__(self, agg_funcs: Union[str, List[str]] = ['mean'],
#                 n_folds: int = 5, cv_strategy: str = 'kfold', 
#                 shuffle: bool = True, random_state: Optional[int] = 42,
#                 drop_original: bool = True, smooth: Union[float, str] = 'auto', 
#                 min_samples_leaf: int = 1, handle_unknown: str = 'global_mean',
#                 handle_missing: str = 'global_mean', verbose: int = 0):
#         self.agg_funcs = agg_funcs
#         self.n_folds = n_folds
#         self.cv_strategy = cv_strategy
#         self.shuffle = shuffle
#         self.random_state = random_state
#         self.drop_original = drop_original
#         self.smooth = smooth
#         self.min_samples_leaf = min_samples_leaf
#         self.handle_unknown = handle_unknown
#         self.handle_missing = handle_missing
#         self.verbose = verbose

#         if isinstance(self.agg_funcs, str):
#             self.agg_funcs = [self.agg_funcs]

#         self._validate_agg_funcs()

#     def _validate_agg_funcs(self):
#         valid_aggs = [
#             'mean', 'std', 'skew', 'kurt', 'sum', 'count',
#             'min', 'max', 'median', 'var', 'sem', 'mode',
#             'quantile_25', 'quantile_75', 'iqr', 'range',
#             'nunique', 'entropy', 'cv', 'mad'
#         ]

#         for agg in self.agg_funcs:
#             if agg not in valid_aggs:
#                 raise ValueError(f'Invalid aggregation function: {agg}. 'f'Valid options: {valid_aggs}')

#     def _compute_all_aggregations(self, group_series, target_series):
#         results = {}

#         if 'mean' in self.agg_funcs:
#             results['mean'] = target_series.mean()

#         if 'std' in self.agg_funcs:
#             results['std'] = target_series.std() if len(target_series) > 1 else 0

#         if 'var' in self.agg_funcs:
#             results['var'] = target_series.var() if len(target_series) > 1 else 0
        
#         if 'sem' in self.agg_funcs:
#             results['sem'] = target_series.sem() if len(target_series) > 1 else 0
        
#         if 'skew' in self.agg_funcs and len(target_series) > 2:
#             try:
#                 from scipy.stats import skew
#                 results['skew'] = skew(target_series)
#             except (ImportError, ValueError):
#                 results['skew'] = 0
        
#         if 'kurt' in self.agg_funcs and len(target_series) > 3:
#             try:
#                 from scipy.stats import kurtosis
#                 results['kurt'] = kurtosis(target_series)
#             except (ImportError, ValueError):
#                 results['kurt'] = 0
        
#         if 'sum' in self.agg_funcs:
#             results['sum'] = target_series.sum()
        
#         if 'count' in self.agg_funcs:
#             results['count'] = len(target_series)
        
#         if 'min' in self.agg_funcs:
#             results['min'] = target_series.min()
        
#         if 'max' in self.agg_funcs:
#             results['max'] = target_series.max()
        
#         if 'median' in self.agg_funcs:
#             results['median'] = target_series.median()
        
#         if 'mode' in self.agg_funcs:
#             mode_val = target_series.mode()
#             results['mode'] = mode_val.iloc[0] if not mode_val.empty else target_series.mean()
        
#         if 'quantile_25' in self.agg_funcs:
#             results['quantile_25'] = target_series.quantile(0.25)
        
#         if 'quantile_75' in self.agg_funcs:
#             results['quantile_75'] = target_series.quantile(0.75)
        
#         if 'iqr' in self.agg_funcs:
#             q1 = target_series.quantile(0.25)
#             q3 = target_series.quantile(0.75)
#             results['iqr'] = q3 - q1
        
#         if 'range' in self.agg_funcs:
#             results['range'] = target_series.max() - target_series.min()
        
#         if 'nunique' in self.agg_funcs:
#             results['nunique'] = target_series.nunique()
        
#         if 'entropy' in self.agg_funcs:
#             # For binary classification entropy
#             from scipy.stats import entropy
#             value_counts = target_series.value_counts(normalize=True)
#             results['entropy'] = entropy(value_counts)
        
#         if 'cv' in self.agg_funcs:  # Coefficient of variation
#             mean_val = target_series.mean()
#             std_val = target_series.std()
#             results['cv'] = std_val / mean_val if mean_val != 0 else 0
        
#         if 'mad' in self.agg_funcs:  # Mean absolute deviation
#             results['mad'] = (target_series - target_series.mean()).abs().mean()
        
#         return results

#     def fit(self, X, y, columns=None):
#         X = X.copy()
#         y = pd.Series(y).reset_index(drop=True)
#         X.reset_index(drop=True, inplace=True)

#         if columns is None:
#             self.features_ = X.select_dtypes(include=['object', 'category']).columns.tolist()

#         else:
#             self.features_ = columns

#         if len(self.features_) == 0:
#             warnings.warn('No categorical columns found for encoding ...')
#             return self

#         self.global_stats_ = {}

#         for agg in self.agg_funcs:
#             if agg == 'mean':
#                 self.global_stats_['mean'] = y.mean()
#             elif agg == 'std':
#                 self.global_stats_['std'] = y.std()
#             elif agg == 'skew':
#                 self.global_stats_['skew'] = y.skew() if hasattr(y, 'skew') else 0
#             elif agg == 'sum':
#                 self.global_stats_['sum'] = y.sum()
#             elif agg == 'count':
#                 self.global_stats_['count'] = len(y)
#             elif agg == 'min':
#                 self.global_stats_['min'] = y.min()
#             elif agg == 'max':
#                 self.global_stats_['max'] = y.max()
#             elif agg == 'median':
#                 self.global_stats_['median'] = y.median()

#             # in process ....

#         self.encodings_ = {col: {} for col in self.features_}
#         for col in self.features_:
#             df = pd.DataFrame({col: X[col], 'target': y})

#             # if df[col].isnull().any():
#             #     if self.hande

#             grouped = df.groupby(col)

#             for category, group in grouped:
#                 if len(group) < self.min_samples_leaf:
#                     continue

#                 agg_results = self._compute_all_aggregations(group[col], group['target'])

#                 if 'mean' in agg_results and self.smooth != 0:
#                     smooth_factor = self._compute_smooth_factor(len(group), len(df))
#                     agg_results['mean'] = (
#                         smooth_factor * agg_results['mean'] + (1 - smooth_factor) * self.global_stats_.get('mean', 0)
#                     )

#                 for agg_name, agg_value in agg_results.items():
#                     if agg_name not in self.encodings_[col]:
#                         self.encodings_[col][agg_name] = {}
#                     self.encodings_[col][agg_name][category] = agg_value

#         self.is_fitted_ = True
#         if self.verbose > 0:
#             print(f'Fitted encoder on {len(self.features_)} columns with aggregations: {self.agg_funcs}')
#             return self

#     def transform(self, X, y=None):
#         check_is_fitted(self, 'is_fitted_')
        
#         X = X.copy()
#         if len(self.features_) == 0:
#             return X
        
#         for col in self.features_:
#             if col not in X.columns:
#                 continue  # Fixed typo
            
#             for agg_name in self.agg_funcs:
#                 new_col_name = f"{col}_{agg_name}"
                
#                 # Get encoding map for this aggregation
#                 encoding_map = self.encodings_[col].get(agg_name, {})
                
#                 def map_value(val):
#                     if pd.isna(val):
#                         return self._handle_missing_value(agg_name)
                    
#                     if val in encoding_map:
#                         return encoding_map[val]
                    
#                     return self._handle_unknown_value(agg_name)
                
#                 X[new_col_name] = X[col].apply(map_value)
        
#         if self.drop_original:
#             X = X.drop(columns=self.features_)
        
#         if self.verbose > 0:
#             print(f"Created {len(self.agg_funcs)} features per categorical column")
        
#         return X

#     def fit_transform(self, X, y, columns=None):
#         """Fit and transform with cross-validation"""
#         X = X.copy().reset_index(drop=True)
#         y = pd.Series(y).reset_index(drop=True)
        
#         if columns is None:
#             features = X.select_dtypes(include=['object', 'category']).columns.tolist()
#         else:
#             features = columns
        
#         if len(features) == 0:
#             return X
        
#         # Initialize result dataframe
#         result_df = X.copy()
        
#         # Pre-create all aggregation columns with proper dtype
#         for col in features:
#             for agg_name in self.agg_funcs:
#                 result_df[f"{col}_{agg_name}"] = pd.Series(dtype='float64')
        
#         # Get CV splits
#         cv = self._get_cv(y)
        
#         if self.verbose > 0:
#             print(f"Performing {self.n_folds}-fold CV encoding...")
        
#         # Perform CV encoding
#         for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
#             if self.verbose > 0:
#                 print(f"  Processing fold {fold}/{self.n_folds}")
            
#             X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#             y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
#             # Fit on training fold
#             fold_encoder = copy.deepcopy(self)
#             fold_encoder.fit(X_train, y_train, columns=features)
            
#             # Transform validation fold
#             val_encoded = fold_encoder.transform(X_val)
            
#             # Fill in the result dataframe with proper type conversion
#             for col in features:
#                 for agg_name in self.agg_funcs:
#                     new_col = f"{col}_{agg_name}"
#                     # Convert to float to ensure consistent dtype
#                     values = pd.to_numeric(val_encoded[new_col], errors='coerce').values
#                     result_df.loc[val_idx, new_col] = values
        
#         # Fit on full data for future transforms
#         self.fit(X, y, columns=features)
        
#         if self.drop_original:
#             result_df = result_df.drop(columns=features)
        
#         if self.verbose > 0:
#             print(f"Total features created: {len(features) * len(self.agg_funcs)}")
        
#         return result_df
    
#     # Helper methods (same as before, but included for completeness)
#     def _get_cv(self, y=None):
#         if self.cv_strategy == 'stratified' and y is not None:
#             return StratifiedKFold(n_splits=self.n_folds, shuffle=self.shuffle, 
#                                  random_state=self.random_state)
#         return KFold(n_splits=self.n_folds, shuffle=self.shuffle, 
#                     random_state=self.random_state)
    
#     def _compute_smooth_factor(self, category_count, n_samples):
#         if self.smooth == 'auto':
#             return n_samples / (n_samples + category_count)
#         elif isinstance(self.smooth, (int, float)):
#             return self.smooth
#         else:
#             return 0.5
    
#     def _handle_missing_value(self, agg_name):
#         if self.handle_missing == 'global_mean':
#             return self.global_stats_.get(agg_name, 0)
#         else:
#             return np.nan
    
#     def _handle_unknown_value(self, agg_name):
#         if self.handle_unknown == 'global_mean':
#             return self.global_stats_.get(agg_name, 0)
#         elif self.handle_unknown == 'error':
#             raise ValueError(f"Unknown category encountered")
#         else:
#             return np.nan
    
#     def get_feature_names_out(self, input_features=None):
#         """Get output feature names"""
#         output_features = []
        
#         for col in self.features_:
#             for agg_name in self.agg_funcs:
#                 output_features.append(f"{col}_{agg_name}")
        
#         if input_features is not None:
#             non_encoded = [f for f in input_features if f not in self.features_]
#             output_features.extend(non_encoded)
        
#         return output_features

In [19]:
xgb_params = {
    'n_estimators': 10000,
    'learning_rate': 0.005,
    'subsample': 0.8,
    # 'colsample_by_tree': 0.7,
    # 'sampling_method': 'gradient_based',
    # 'reg_alpha': 2.0,
    # 'reg_lambda': 4.0,
    'eval_metric': 'auc',
    'enable_categorical': True,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

lgb_params = {
    'n_estimators': 10_000,
    'learning_rate': 0.005,
    'max_depth': 8,                 # same philosophy
    'num_leaves': 2 ** 8,           # typical rule: ≤ 2^max_depth
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    # 'reg_lambda': 4.0,
    'random_state': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'auc',
    'n_jobs': -1,
    # 'verbose': 200,
    'verbosity':-1,
    # 'device': 'cuda'
    # 'cat_feature': CATS,            # pass categorical indices/names
}

cat_params = {
    'iterations': 10_000,          # same as n_estimators
    'learning_rate': 0.005,
    'depth': 8,                    # max_depth equivalent
    'subsample': 0.8,              # bagging
    'colsample_bylevel': 0.7,      # feature fraction per split
    'reg_lambda': 4.0,             # L2
    'random_seed': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'AUC',
    'thread_count': -1,            # use all cores
    'verbose': 200,                # same as XGB verbose
    # 'verbosity':-1
    # 'cat_features': CATS,          # list of column names / indices
}

real_mlp_params = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'val_metric_name': '1-auc_ovo',
        'n_epochs': 60,
        'batch_size': 1024,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }

In [20]:
X.dtypes

Age                               int32
Sex                               int32
Chest pain type                   int32
BP                                int32
Cholesterol                       int32
FBS over 120                      int32
EKG results                       int32
Max HR                            int32
Exercise angina                   int32
ST depression                   float32
Slope of ST                       int32
Number of vessels fluro           int32
Thallium                          int32
Age_cat                        category
Sex_cat                        category
Chest pain type_cat            category
BP_cat                         category
Cholesterol_cat                category
FBS over 120_cat               category
EKG results_cat                category
Max HR_cat                     category
Exercise angina_cat            category
ST depression_cat              category
Slope of ST_cat                category
Number of vessels fluro_cat    category


In [21]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test_n = X_test.copy()
    # X_org_n = X_org.copy()
    # y_org_n = y_org.copy()
    # X_org_n = pd.concat([X_org_n]*20, axis=0, ignore_index=True)
    # y_org_n = pd.concat([y_org_n]*20, axis=0, ignore_index=True)
    # X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    # y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    # model = clone(xgb.XGBClassifier(**xgb_params))
    # model = clone(cb.CatBoostClassifier(**cat_params))
    # model = clone(lgb.LGBMClassifier(**lgb_params))

    # model = clone(RealMLP_TD_Classifier(**real_mlp_params))
    for c in CATS:
        n = f'{c}_mean_te'
        TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
        X_train[n] = TE.fit_transform(pd.DataFrame(X_train[c]), y_train).flatten()
        X_val[n] = TE.transform(pd.DataFrame(X_val[c])).flatten()
        X_test_n[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
    # for col in INTER:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
        
    # TE = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=False, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE.fit_transform(X_train, y_train, columns=CATS)
    # X_val = TE.transform(X_val)
    # X_test_n = TE.transform(X_test_n)

    # TE1 = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=True, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE1.fit_transform(X_train, y_train, columns=INTER)
    # X_val = TE1.transform(X_val)
    # X_test_n = TE1.transform(X_test_n)

    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str).astype('category')
    #     X_val[col] = X_val[col].astype(str).astype('category')
    #     X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str).astype('category')
    #     X_val[col] = X_val[col].astype(str).astype('category')
    #     X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # X_train.drop(columns=INTER, inplace=True)
    # X_val.drop(columns=INTER, inplace=True)
    # X_test_n.drop(columns=INTER, inplace=True)
    # print(X_train.dtypes)
    print(X_train.shape)
    param_grid = {'colsample_bytree': 0.2364,
                  'gamma': 0.034283,
                  'max_depth': 6,
                  'reg_alpha': 0.71367,
                  'reg_lambda': 4.43564,
                  'subsample': 0.59394}

    model = xgb.XGBClassifier(**param_grid,
                          n_estimators=10000,
                          objective='binary:logistic',
                          eval_metric='auc',
                          learning_rate=0.01,
                          early_stopping_rounds=500,
                          max_bin=1024,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)
    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=500)
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test_n)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

(504000, 39)
[0]	validation_0-auc:0.91716
[500]	validation_0-auc:0.95559
[1000]	validation_0-auc:0.95598
[1500]	validation_0-auc:0.95595
[1646]	validation_0-auc:0.95593
SCORE FOR FOLD1 : 0.955979683085828
(504000, 39)
[0]	validation_0-auc:0.91566
[500]	validation_0-auc:0.95451
[1000]	validation_0-auc:0.95486
[1500]	validation_0-auc:0.95484
[1514]	validation_0-auc:0.95484
SCORE FOR FOLD2 : 0.9548619454234245
(504000, 39)
[0]	validation_0-auc:0.91597
[500]	validation_0-auc:0.95532
[1000]	validation_0-auc:0.95571
[1500]	validation_0-auc:0.95571
[1634]	validation_0-auc:0.95570
SCORE FOR FOLD3 : 0.9557276116494555
(504000, 39)
[0]	validation_0-auc:0.91580
[500]	validation_0-auc:0.95483
[1000]	validation_0-auc:0.95531
[1500]	validation_0-auc:0.95532
[1777]	validation_0-auc:0.95529
SCORE FOR FOLD4 : 0.9553314424551127
(504000, 39)
[0]	validation_0-auc:0.91799
[500]	validation_0-auc:0.95579
[1000]	validation_0-auc:0.95617
[1500]	validation_0-auc:0.95615
[1582]	validation_0-auc:0.95614
SCORE FO

In [22]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)